In [ ]:
pip install langchain-ollama

In [ ]:
pip install grandalf

In [1]:
from langchain_ollama import ChatOllama
model = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)

# Simple Chain

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate(
    template= "Generate 3 intersting fact about {person}",
    input_variables=["person"]
)

parser = StrOutputParser()

chain = prompt | model | parser
result = chain.invoke({"person": "APJ Abdul Kalam"})
print(result)
chain.get_graph().print_ascii()

Here are three interesting facts about **APJ Abdul Kalam**:

1. **First Indian President with a PhD**: APJ Abdul Kalam was the first Indian president to hold a PhD degree in engineering. He earned his doctorate in aeronautics from the Indian Institute of Technology, Kharagpur, in 1963, making him a pioneer in India's scientific and educational landscape.

2. **Pilot and Scientist**: Before becoming India's president, Kalam was a test pilot and engineer. He famously flew the **Agni** missile, India's first intercontinental ballistic missile, during his military career, showcasing his expertise in aerospace technology.

3. **Cricket Enthusiast**: Despite his monumental achievements, Kalam was a passionate cricket fan. He often played cricket in his later years, and his love for the sport was a testament to his humble, patriotic spirit. He even wrote a book titled *The Cricket Cricketer* about his experiences playing the game. 

These facts highlight his dual legacy as a visionary leader 

### Sequencial Chain

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Slogans(BaseModel):
    slogans: list[str] = Field(description="List of marketing slogans", min_length=5, max_length=5)

parser = PydanticOutputParser(pydantic_object=Slogans)

prompt1 = PromptTemplate(
    template= "Generate a creative product description for a product : {product_name}",
    input_variables=["product_name"])

prompt2 = PromptTemplate(
    template= "Generate 5 unique and catchy marketing slogans for a product description: \n {product_description}\n{format_instructions}",
    input_variables=["product_description"],
    partial_variables={"format_instructions": parser.get_format_instructions()})

chain = prompt1 | model | prompt2 | model | parser
result = chain.invoke({"product_name": "Samsung Galaxy S23 Ultra Smartphone with S-Pen"})
for slogan in result.slogans:
    print("-", slogan)
chain.get_graph().print_ascii()

- S-Pen 2.0: Your Digital Canvas for Creativity
- Immerse Yourself in Infinite Detail with 120Hz AMOLED
- Unleash Power, Creativity, and Precision with the S23 Ultra
- 5G Speed, AI Intelligence – Transform Your Work
- Elevate Your Visuals with 120Hz AMOLED Display
    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOllama |      
     +------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOllama |      
     +------------+      
            *            
       

### Parallel Chaining 

In [15]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableParallel

model1 = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)

model2 = ChatOllama(
    model="qwen3-vl:8b",
    temperature=0,
)
prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)
prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)
prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)
parser = StrOutputParser()
parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})
text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""
merge_chain = prompt3 | model1 | parser
chain = parallel_chain | merge_chain

result = chain.invoke({"text": text})
print(result)
chain.get_graph().print_ascii()

Notes:  
1. What are SVMs? Supervised learning methods for classification, regression, and outliers detection.  
2. Advantages:  
   - Effective in high-dimensional spaces.  
   - Handles more dimensions than samples.  
   - Uses support vectors (memory-efficient).  
   - Versatile with customizable kernels.  
3. Disadvantages:  
   - High feature dimensions require careful kernel/regularization selection.  
   - No direct probability estimates (calculated via cross-validation).  
4. Scikit-learn Support:  
   - Supports dense (numpy) and sparse (scipy) data.  
   - Predictions on sparse data require training on sparse data.  
   - Optimal performance with C-ordered dense arrays or sparse CSR matrices (float64).  

Quiz:  
1. Q: What are SVMs primarily used for? A: Classification, regression, and outliers detection.  
2. Q: Why are SVMs effective in high-dimensional spaces? A: They remain effective even when the number of dimensions exceeds the number of samples.  
3. Q: How does SVM a

### Conditional Chain

In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableParallel,RunnableLambda,RunnableBranch
from typing import Literal
model = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)
parser = StrOutputParser()

class Feedback(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)
classification_chain = prompt1 | model | parser2
prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)
prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)
branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x: x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "No valid sentiment found")
)
chain = classification_chain | branch_chain
print(chain.invoke({
    "feedback": "The product quality is excellent and delivery was prompt. Very satisfied with my purchase!"
}))
chain.get_graph().print_ascii()

You're amazing! Thank you so much for your kind words! 😊 I'm really glad you think so. Let me know if there's anything else I can do for you!
    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOllama |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
